# Pipeline LOSO para detección y predicción de Freezing of Gait (FoG) con sensores wearables - Dataset Figshare

**Flujo:**
1. Cargar dataset procesado
2. Anotar etiquetas (binario + multiclase con ventana pre-FoG)
3. Crear ventanas deslizantes por sujeto/trial
4. Detectar/interpolar outliers en ventanas crudas
5. Rellenar valores faltantes (interpolación polinomial como fallback)
6. Extraer características determinísticas por ventana
7. Pipeline LOSO por fold: limpiar, extraer features, escalar (solo en train), SMOTE (solo en train), guardar CSVs por fold
8. Visualizaciones clave para QA y presentación a inversores

**Características extraídas por ventana:**
- Tiempo: media, std, skew, kurtosis, RMS, mediana, IQR
- Derivadas: media/std de magnitud, cadencia (picos)
- Frecuencia: pico PSD, energía total PSD, potencias de banda (0.5-3 Hz, 3-8 Hz), índice de freezing
- Wavelet: energías, entropía
- No lineales: entropía de muestra, dimensión fractal de Higuchi

In [ ]:
# Configuración global y librerías
WINDOW_SIZE_SEC = 4
OVERLAP = 0.5

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import signal, stats
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from imblearn.over_sampling import SMOTE
import pywt

# Opcional: para características no lineales
try:
    from nolds import sampen, higuchi_fd
except ImportError:
    sampen = None
    higuchi_fd = None

# Ruta base de datos y outputs
DATA_PATH = Path('../../Datasets/Figshare a public dataset')
OUTPUT_PATH = Path('../../outputs/figshare_features')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

SAMPLING_RATE = 100  # Ajustar según dataset real

## 1. Cargar dataset procesado
Cargamos las señales preprocesadas y metadatos (sujetos, trials, frecuencia de muestreo).

In [ ]:
# Cargar dataset procesado (ajustar ruta y columnas según estructura real)
df = pd.read_csv(DATA_PATH / 'PDFEinfo_cleaned.csv')
print('Shape:', df.shape)
df.head()

## 2. Anotar etiquetas (binario y multiclase con ventana pre-FoG)
Generamos etiquetas binario (FoG vs no-FoG) y multiclase, agregando una ventana pre-FoG de 0.5 * SAMPLING_RATE.

In [ ]:
# Etiquetado binario y multiclase con ventana pre-FoG
def annotate_labels(df, fog_col='FoG', pre_fog_sec=2, sr=SAMPLING_RATE):
    df = df.copy()
    pre_fog_samples = int(pre_fog_sec * sr)
    fog_idx = df.index[df[fog_col] == 1].tolist()
    pre_fog_idx = []
    for idx in fog_idx:
        pre_fog_idx.extend(range(max(0, idx - pre_fog_samples), idx))
    df['label_binary'] = 0
    df.loc[fog_idx, 'label_binary'] = 1
    df.loc[pre_fog_idx, 'label_binary'] = 2  # 2 = pre-FoG
    # Multiclase: 0 = no-FoG, 1 = pre-FoG, 2 = FoG
    df['label_multi'] = 0
    df.loc[pre_fog_idx, 'label_multi'] = 1
    df.loc[fog_idx, 'label_multi'] = 2
    return df

# Aplicar anotación
df = annotate_labels(df, fog_col='FoG', pre_fog_sec=0.5, sr=SAMPLING_RATE)
df[['label_binary', 'label_multi']].value_counts().plot.bar()
plt.title('Distribución de etiquetas (binario/multiclase)')
plt.show()

# Pipeline LOSO para detección y predicción de Freezing of Gait (FoG) con sensores wearables - Dataset Figshare

**Flujo:**
- Cargar dataset procesado
- Anotar etiquetas (binario + multiclase con ventana pre-FoG)
- Crear ventanas deslizantes por sujeto/trial
- Detectar/interpolar outliers en ventanas crudas
- Rellenar valores faltantes (interpolación polinomial como fallback)
- Extraer características determinísticas por ventana
- Pipeline LOSO por fold: limpiar, extraer features, escalar (solo en train), SMOTE (solo en train), guardar CSVs por fold

**Características extraídas por ventana:**
- Tiempo: media, std, skew, kurtosis, RMS, mediana, IQR
- Derivadas: media/std de magnitud, cadencia (picos)
- Frecuencia: pico PSD, energía total PSD, potencias de banda (0.5-3 Hz, 3-8 Hz), índice de freezing
- Wavelet: energías, entropía
- No lineales: entropía de muestra, dimensión fractal de Higuchi

In [ ]:
# Configuración global y librerías
WINDOW_SIZE_SEC = 4
OVERLAP = 0.5

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import signal, stats, interpolate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from imblearn.over_sampling import SMOTE
import pywt
try:
    from nolds import sampen, higuchi_fd
except ImportError:
    sampen = None; higuchi_fd = None

DATA_PATH = Path('../../Datasets/Figshare a public dataset')
OUTPUT_PATH = Path('../../outputs/figshare_features')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
SAMPLING_RATE = 100  # Ajustar según dataset real

In [ ]:
# --- Robust window creation (override) to avoid MemoryError on concatenation ---
def create_windows_per_subject_safe(df_subset, feature_cols, label_col, window_size, step_size):
    all_windows = []
    all_labels = []
    all_subjects = []

    for (subject, trial), group in df_subset.groupby(['subject', 'trial']):
        data = group[feature_cols].values
        labels = group[label_col].values
        if len(data) < window_size:
            continue
        # Slide windows for this subject-trial
        local_windows = []
        local_labels = []
        for start in range(0, len(data) - window_size + 1, step_size):
            end = start + window_size
            window = data[start:end]
            window_label = np.bincount(labels[start:end]).argmax()
            local_windows.append(window)
            local_labels.append(window_label)
        if len(local_windows) == 0:
            continue
        all_windows.append(np.array(local_windows))
        all_labels.append(np.array(local_labels))
        all_subjects.append(np.array([subject] * len(local_windows)))

    # Filter empties
    all_windows = [w for w in all_windows if getattr(w, 'size', 0) > 0]
    all_labels = [l for l in all_labels if getattr(l, 'size', 0) > 0]
    all_subjects = [s for s in all_subjects if getattr(s, 'size', 0) > 0]

    if len(all_windows) == 0:
        return np.empty((0, window_size, len(feature_cols))), np.empty((0,), dtype=int), np.empty((0,), dtype=object)

    X = np.concatenate(all_windows, axis=0)
    y = np.concatenate(all_labels, axis=0)
    subjects = np.concatenate(all_subjects, axis=0)
    return X, y, subjects

# Monkeypatch the name used earlier if needed
create_windows_per_subject = create_windows_per_subject_safe

# --- Per-fold streaming feature extraction and LOSO processing ---
from tqdm import tqdm

def extract_features_from_array(window_array, sr=SAMPLING_RATE):
    # window_array: (window_size, n_channels) pandas-like columns assumed to be numeric order
    # returns a dict of features
    feats = {}
    for ch in range(window_array.shape[1]):
        x = window_array[:, ch]
        feats[f'ch{ch}_mean'] = np.nanmean(x)
        feats[f'ch{ch}_std'] = np.nanstd(x)
        feats[f'ch{ch}_skew'] = stats.skew(x, nan_policy='omit')
        feats[f'ch{ch}_kurt'] = stats.kurtosis(x, nan_policy='omit')
        feats[f'ch{ch}_rms'] = np.sqrt(np.nanmean(x**2))
        feats[f'ch{ch}_median'] = np.nanmedian(x)
        feats[f'ch{ch}_iqr'] = stats.iqr(x, nan_policy='omit')
        dx = np.diff(x)
        feats[f'ch{ch}_dx_mean'] = np.nanmean(dx)
        feats[f'ch{ch}_dx_std'] = np.nanstd(dx)
        peaks, _ = signal.find_peaks(x, distance=int(sr/2))
        feats[f'ch{ch}_cadence'] = len(peaks) / (len(x)/sr)
        f, Pxx = signal.welch(x, fs=sr, nperseg=min(256, len(x)))
        feats[f'ch{ch}_psd_peak'] = f[np.argmax(Pxx)] if len(Pxx) else np.nan
        feats[f'ch{ch}_psd_energy'] = np.nansum(Pxx)
        feats[f'ch{ch}_band_0_3'] = np.nansum(Pxx[(f>=0.5)&(f<3)])
        feats[f'ch{ch}_band_3_8'] = np.nansum(Pxx[(f>=3)&(f<8)])
        feats[f'ch{ch}_freezing_index'] = feats[f'ch{ch}_band_3_8'] / (feats[f'ch{ch}_band_0_3'] + 1e-9)
        try:
            coeffs = pywt.wavedec(x, 'db4', level=3)
            for i, c in enumerate(coeffs):
                feats[f'ch{ch}_wav_energy_L{i}'] = np.nansum(np.square(c))
            feats[f'ch{ch}_wav_entropy'] = stats.entropy(np.abs(np.concatenate(coeffs)) + 1e-9)
        except Exception:
            feats[f'ch{ch}_wav_energy_L0'] = np.nan
            feats[f'ch{ch}_wav_entropy'] = np.nan
        if sampen is not None:
            try:
                feats[f'ch{ch}_sampen'] = sampen(x)
            except Exception:
                feats[f'ch{ch}_sampen'] = np.nan
        if higuchi_fd is not None:
            try:
                feats[f'ch{ch}_hfd'] = higuchi_fd(x)
            except Exception:
                feats[f'ch{ch}_hfd'] = np.nan
    return feats


def process_loso_pipeline_streaming(df, feature_cols, label_col='label_binary', window_sec=WINDOW_SIZE_SEC, overlap=OVERLAP, sr=SAMPLING_RATE):
    window_size = int(window_sec * sr)
    step_size = int(window_size * (1-overlap))
    logo = LeaveOneGroupOut()
    groups = df['subject'].values
    summaries = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(df, df[label_col], groups)):
        print(f'Processing fold {fold}...')
        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_test = df.iloc[test_idx].reset_index(drop=True)

        # Create windows safely
        X_train_win, y_train_win, subjects_train_win = create_windows_per_subject_safe(df_train, feature_cols, label_col, window_size, step_size)
        X_test_win, y_test_win, subjects_test_win = create_windows_per_subject_safe(df_test, feature_cols, label_col, window_size, step_size)

        print(f' Fold {fold}: train windows={len(X_train_win)}, test windows={len(X_test_win)}')

        # Extract features streaming to lists (memory-efficient)
        train_feats = []
        for w in tqdm(X_train_win, desc=f'fold{fold}-train-feat'):
            train_feats.append(extract_features_from_array(w, sr=sr))
        test_feats = []
        for w in tqdm(X_test_win, desc=f'fold{fold}-test-feat'):
            test_feats.append(extract_features_from_array(w, sr=sr))

        if len(train_feats) == 0:
            print(f' Fold {fold} has no train features, skipping')
            continue

        X_train_df = pd.DataFrame(train_feats)
        X_test_df = pd.DataFrame(test_feats)
        y_train = pd.Series(y_train_win, name='label')
        y_test = pd.Series(y_test_win, name='label')

        # Impute and scale (fit only on train)
        imp = SimpleImputer(strategy='median')
        X_train_imp = pd.DataFrame(imp.fit_transform(X_train_df), columns=X_train_df.columns)
        X_test_imp = pd.DataFrame(imp.transform(X_test_df.reindex(columns=X_train_df.columns, fill_value=np.nan)), columns=X_train_df.columns)
        scaler = StandardScaler().fit(X_train_imp.values)
        X_train_scaled = scaler.transform(X_train_imp.values)
        X_test_scaled = scaler.transform(X_test_imp.values)

        # SMOTE on train only
        sm = SMOTE(random_state=42)
        try:
            X_res, y_res = sm.fit_resample(X_train_scaled, y_train)
        except Exception as e:
            print(f' Fold {fold} SMOTE failed: {e} - using original train')
            X_res, y_res = X_train_scaled, y_train

        # Save per-fold CSVs
        fold_dir = OUTPUT_PATH / f'fold_{fold}_subj_{np.unique(df_test["subject"])[0]}'
        fold_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(X_res, columns=X_train_df.columns).to_csv(fold_dir / 'X_train_resampled.csv', index=False)
        pd.Series(y_res, name='label').to_csv(fold_dir / 'y_train_resampled.csv', index=False)
        pd.DataFrame(X_test_scaled, columns=X_train_df.columns).to_csv(fold_dir / 'X_test.csv', index=False)
        pd.Series(y_test.values, name='label').to_csv(fold_dir / 'y_test.csv', index=False)

        summaries.append({'fold': fold, 'test_subject': np.unique(df_test['subject'])[0], 'train_resampled': len(y_res), 'test_size': len(y_test)})

        # Quick visualizations per fold (class balance)
        try:
            fig, ax = plt.subplots(1,2, figsize=(8,3))
            ax[0].bar(['no-FoG','FoG'], [sum(y_train==0), sum(y_train==1)])
            ax[0].set_title('Train before SMOTE')
            ax[1].bar(['no-FoG','FoG'], [sum(y_res==0), sum(y_res==1)])
            ax[1].set_title('Train after SMOTE')
            plt.suptitle(f'Fold {fold} class balance')
            plt.tight_layout(); plt.show()
        except Exception:
            pass

    return pd.DataFrame(summaries)

# Run full pipeline (callable cell)
print('Ready: call process_loso_pipeline_streaming(df, feature_cols) to execute full LOSO pipeline per-fold.')


## Funciones: Cargar y preparar dataset

Cada función está en su propia celda y sigue el estilo del notebook `daphnet_segmentation_reordered.ipynb`.

In [ ]:
def load_processed_figshare(csv_path=None, subject_col='subject_id', session_col='session_id', time_col='time_s'):
    """Load processed figshare CSV and normalize column names. Returns DataFrame and feature column list."""
    path = DATA_PATH / 'PDFEinfo_cleaned.csv' if csv_path is None else Path(csv_path)
    df = pd.read_csv(path)
    # Normalize subject/trial columns
    if subject_col in df.columns:
        df['subject'] = 'S' + df[subject_col].astype(str).str.zfill(2)
    if session_col in df.columns:
        df['trial'] = 'R' + df[session_col].astype(str).str.zfill(2)
    # detect sensor columns heuristically (common names)
    candidate = ['acc_ml_g','acc_ap_g','acc_si_g','gyr_ml_deg_s','gyr_ap_deg_s','gyr_si_deg_s']
    feature_cols = [c for c in candidate if c in df.columns]
    # fallback: numerical columns excluding labels/meta
    if len(feature_cols) == 0:
        exclude = {subject_col, session_col, 'label', 'FoG', 'time_s'}
        feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
    print(f'Loaded {len(df):,} rows; detected {len(feature_cols)} sensor cols')
    return df, feature_cols

# Example usage (cell can be run):
# df, feature_cols = load_processed_figshare()

In [ ]:
def annotate_labels_df(df, fog_col='FoG', pre_fog_sec=0.5, sr=SAMPLING_RATE):
    """Create binary and multiclass labels with pre-FoG buffer (returns new df copy)."""
    df = df.copy()
    pre_fog_samples = int(pre_fog_sec * sr)
    # Binary: assume fog_col is 1 for FoG
    df['label_binary'] = 0
    if fog_col in df.columns:
        fog_idx = df.index[df[fog_col] == 1].tolist()
    else:
        fog_idx = []
    pre_idx = []
    for idx in fog_idx:
        pre_idx.extend(range(max(0, idx - pre_fog_samples), idx))
    if fog_idx:
        df.loc[fog_idx, 'label_binary'] = 1
    # Mark pre-FoG as separate code (2) if desired
    df['label_multi'] = df['label_binary']
    if len(pre_idx) > 0:
        mask = df.index.isin(pre_idx) & (df['label_multi'] == 0)
        df.loc[mask, 'label_multi'] = 2
    print('Labels: ', df['label_binary'].value_counts().to_dict())
    print('Multiclass: ', df['label_multi'].value_counts().to_dict())
    return df

# Example: df = annotate_labels_df(df, fog_col='FoG', pre_fog_sec=0.5)

In [ ]:
def create_windows_per_subject_df(df_subset, feature_cols, label_col, window_sec=WINDOW_SIZE_SEC, overlap=OVERLAP, sr=SAMPLING_RATE):
    """Create sliding windows per subject/trial and return list of dicts (one window per dict).

    This preserves DataFrame columns and avoids concatenating empty lists.
    """
    window_size = int(window_sec * sr)
    step = int(window_size * (1-overlap))
    windows = []
    for (subject, trial), group in df_subset.groupby(['subject', 'trial']):
        data = group.reset_index(drop=True)
        n = len(data)
        if n < window_size:
            continue
        for start in range(0, n - window_size + 1, step):
            end = start + window_size
            w = data.iloc[start:end].copy()
            label_bin = int(w[label_col].mode().iat[0]) if label_col in w.columns else 0
            windows.append({'subject': subject, 'trial': trial, 'start': start, 'end': end, 'window': w, 'label_binary': label_bin})
    return windows

# Example: win = create_windows_per_subject_df(df_train, feature_cols, 'label_binary')

In [ ]:
def interpolate_outliers_df(window_df, z_thresh=4.0):
    """Detect outliers per numeric column using z-score and interpolate them (linear).

    Returns cleaned DataFrame (same index length).
    """
    w = window_df.copy()
    num_cols = w.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        vals = w[col].values.astype(float)
        if len(vals) < 3:
            continue
        z = np.abs(stats.zscore(vals, nan_policy='omit'))
        outliers = np.where(z > z_thresh)[0]
        if outliers.size:
            good = np.setdiff1d(np.arange(len(vals)), outliers)
            if good.size > 1:
                interp = interpolate.interp1d(good, vals[good], kind='linear', fill_value='extrapolate')
                vals[outliers] = interp(outliers)
                w[col] = vals
    return w

# Example: w_clean = interpolate_outliers_df(window['window'])

In [ ]:
def fill_missing_poly_df(window_df, order=3):
    """Fill missing values in DataFrame window using linear then polynomial fallback."""
    w = window_df.copy()
    num_cols = w.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if w[col].isnull().any():
            idx = np.arange(len(w))
            not_nan = ~w[col].isnull()
            if not_nan.sum() > order:
                try:
                    poly = np.polyfit(idx[not_nan], w[col].values[not_nan], order)
                    w[col] = np.where(not_nan, w[col], np.polyval(poly, idx))
                except Exception:
                    w[col] = w[col].interpolate(method='linear', limit_direction='both')
            else:
                w[col] = w[col].interpolate(method='linear', limit_direction='both')
    return w

# Example: w_filled = fill_missing_poly_df(w_clean)

In [ ]:
def extract_features_df(window_df, sr=SAMPLING_RATE):
    """Extract deterministic features from a DataFrame window (per-channel)."""
    feats = {}
    num_cols = window_df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        x = window_df[col].values.astype(float)
        feats[f'{col}_mean'] = np.nanmean(x)
        feats[f'{col}_std'] = np.nanstd(x)
        feats[f'{col}_skew'] = stats.skew(x, nan_policy='omit')
        feats[f'{col}_kurt'] = stats.kurtosis(x, nan_policy='omit')
        feats[f'{col}_rms'] = np.sqrt(np.nanmean(x**2))
        feats[f'{col}_median'] = np.nanmedian(x)
        feats[f'{col}_iqr'] = stats.iqr(x, nan_policy='omit')
        dx = np.diff(x)
        feats[f'{col}_dx_mean'] = np.nanmean(dx) if len(dx)>0 else 0.0
        feats[f'{col}_dx_std'] = np.nanstd(dx) if len(dx)>0 else 0.0
        peaks, _ = signal.find_peaks(x, distance=int(sr/2))
        feats[f'{col}_cadence'] = len(peaks) / (len(x)/sr) if len(x)>0 else 0.0
        try:
            f, Pxx = signal.welch(x, fs=sr, nperseg=min(256, len(x)))
            feats[f'{col}_psd_peak'] = f[np.argmax(Pxx)] if len(Pxx) else np.nan
            feats[f'{col}_psd_energy'] = np.nansum(Pxx)
            feats[f'{col}_band_0_3'] = np.nansum(Pxx[(f>=0.5)&(f<3)])
            feats[f'{col}_band_3_8'] = np.nansum(Pxx[(f>=3)&(f<8)])
            feats[f'{col}_freezing_index'] = feats[f'{col}_band_3_8'] / (feats[f'{col}_band_0_3'] + 1e-9)
        except Exception:
            feats[f'{col}_psd_peak'] = np.nan
            feats[f'{col}_psd_energy'] = np.nan
            feats[f'{col}_band_0_3'] = np.nan
            feats[f'{col}_band_3_8'] = np.nan
            feats[f'{col}_freezing_index'] = np.nan
        # Wavelet
        try:
            coeffs = pywt.wavedec(x, 'db4', level=3)
            for i, c in enumerate(coeffs):
                feats[f'{col}_wav_energy_L{i}'] = np.nansum(np.square(c))
            feats[f'{col}_wav_entropy'] = stats.entropy(np.abs(np.concatenate(coeffs)) + 1e-9)
        except Exception:
            feats[f'{col}_wav_energy_L0'] = np.nan
            feats[f'{col}_wav_entropy'] = np.nan
        # Nonlinear
        if sampen is not None:
            try:
                feats[f'{col}_sampen'] = sampen(x)
            except Exception:
                feats[f'{col}_sampen'] = np.nan
        if higuchi_fd is not None:
            try:
                feats[f'{col}_hfd'] = higuchi_fd(x)
            except Exception:
                feats[f'{col}_hfd'] = np.nan
    return feats

# Example usage: feats = extract_features_df(window['window'])

In [ ]:
def handle_missing_and_scale_df(X_train_df, X_test_df):
    """Impute (median) and scale. Fit on train only. Returns scaled arrays and imputed DataFrames, imputer and scaler."""
    # Coerce to numeric
    X_train_num = X_train_df.select_dtypes(include=[np.number]).copy()
    X_test_num = X_test_df.select_dtypes(include=[np.number]).copy()
    imp = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(imp.fit_transform(X_train_num), columns=X_train_num.columns)
    X_test_imp = pd.DataFrame(imp.transform(X_test_num.reindex(columns=X_train_num.columns, fill_value=np.nan)), columns=X_train_num.columns)
    scaler = StandardScaler().fit(X_train_imp.values)
    X_train_scaled = scaler.transform(X_train_imp.values)
    X_test_scaled = scaler.transform(X_test_imp.values)
    return X_train_scaled, X_test_scaled, X_train_imp, X_test_imp, imp, scaler

# Example: X_train_s, X_test_s, X_train_imp, X_test_imp, imp, scaler = handle_missing_and_scale_df(X_train_df, X_test_df)

In [ ]:
def process_loso_folds_df(df, feature_cols, label_col='label_binary', window_sec=WINDOW_SIZE_SEC, overlap=OVERLAP, sr=SAMPLING_RATE):
    """Full LOSO pipeline: windows created within each fold, cleaned, features extracted, imputed/scaled, SMOTE on train, save per-fold CSVs."""
    window_size = int(window_sec * sr)
    step = int(window_size * (1-overlap))
    logo = LeaveOneGroupOut()
    groups = df['subject'].values
    summaries = []
    for fold, (train_idx, test_idx) in enumerate(logo.split(df, df[label_col], groups)):
        print(f'Fold {fold}: preparing train/test windows...')
        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_test = df.iloc[test_idx].reset_index(drop=True)
        win_train = create_windows_per_subject_df(df_train, feature_cols, label_col, window_sec, overlap, sr)
        win_test = create_windows_per_subject_df(df_test, feature_cols, label_col, window_sec, overlap, sr)
        print(f'  Train windows: {len(win_train)}, Test windows: {len(win_test)}')
        # Process windows -> extract features sequentially (memory-friendly)
        train_feats = []
        for w in win_train:
            w_clean = interpolate_outliers_df(w['window'])
            w_filled = fill_missing_poly_df(w_clean)
            feats = extract_features_df(w_filled, sr=sr)
            feats['subject'] = w['subject']
            feats['trial'] = w['trial']
            feats['label_binary'] = w['label_binary']
            train_feats.append(feats)
        test_feats = []
        for w in win_test:
            w_clean = interpolate_outliers_df(w['window'])
            w_filled = fill_missing_poly_df(w_clean)
            feats = extract_features_df(w_filled, sr=sr)
            feats['subject'] = w['subject']
            feats['trial'] = w['trial']
            feats['label_binary'] = w['label_binary']
            test_feats.append(feats)
        if len(train_feats) == 0:
            print(f'  Fold {fold} has no train features, skipping')
            continue
        X_train_df = pd.DataFrame(train_feats)
        X_test_df = pd.DataFrame(test_feats)
        y_train = X_train_df['label_binary']
        y_test = X_test_df['label_binary']
        X_train = X_train_df.drop(['subject','trial','label_binary'], axis=1, errors='ignore')
        X_test = X_test_df.drop(['subject','trial','label_binary'], axis=1, errors='ignore')
        # Impute & scale
        X_train_scaled, X_test_scaled, X_train_imp, X_test_imp, imp, scaler = handle_missing_and_scale_df(X_train, X_test)
        # SMOTE
        sm = SMOTE(random_state=42)
        try:
            X_res, y_res = sm.fit_resample(X_train_scaled, y_train)
        except Exception as e:
            print(f'  Fold {fold} SMOTE failed: {e} - saving original train')
            X_res, y_res = X_train_scaled, y_train
        # Save
        fold_dir = OUTPUT_PATH / f'fold_{fold}_subj_{np.unique(df_test['subject'])[0]}'
        fold_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(X_res, columns=X_train.columns).to_csv(fold_dir / 'X_train_resampled.csv', index=False)
        pd.Series(y_res, name='label').to_csv(fold_dir / 'y_train_resampled.csv', index=False)
        pd.DataFrame(X_test_scaled, columns=X_train.columns).to_csv(fold_dir / 'X_test.csv', index=False)
        pd.Series(y_test.values, name='label').to_csv(fold_dir / 'y_test.csv', index=False)
        summaries.append({'fold': fold, 'test_subject': np.unique(df_test['subject'])[0], 'train_resampled': len(y_res), 'test_size': len(y_test)})
        # quick summary per fold
        print(f"  Fold {fold}: saved to {fold_dir} (train_resampled={len(y_res)}, test={len(y_test)})")
    return pd.DataFrame(summaries)

# Example: summary = process_loso_folds_df(df, feature_cols)